# Microdrama Promo Finder
Run cells top to bottom. Before starting: (1) add a Colab Secret named `GEMINI_API_KEY` (key icon in left sidebar, enable notebook access); (2) upload `promo_finder.py`, `parse.py` and the screenplay `.zip` to the Colab file panel.

In [ ]:
!pip -q install google-genai

In [ ]:
import os, glob, zipfile
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

# ---- settings you may change ----
os.environ['PROVIDER'] = 'gemini'            # or 'openai'
os.environ['MODEL'] = 'gemini-flash-lite-latest'    # replace with a valid model name from the list below if this errors
os.environ['SLEEP'] = '4'                    # raise to 8-12 if you hit rate-limit errors

z = glob.glob('*.zip')[0]
zipfile.ZipFile(z).extractall('pack')
PACK = os.path.dirname(glob.glob('pack/**/episode-*.txt', recursive=True)[0])
print('Screenplays found in:', PACK, '| count:', len(glob.glob(PACK + '/episode-*.txt')))

In [ ]:
from google import genai
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
for m in client.models.list():
    print(m.name)

## Smoke test on 2 episodes first (about 1 minute)

In [ ]:
import importlib, promo_finder as pf; importlib.reload(pf)
pf.MODEL = os.environ['MODEL']; pf.SLEEP = float(os.environ['SLEEP'])
pf.run(PACK, 'outputs_test', only=[1, 2])
print(open('outputs_test/ranked_promos.md').read()[:2000])


In [ ]:
import json
print(json.dumps(json.load(open('outputs_test/rejected_candidates.json')), indent=1)[:2500])

## Full run on all 50 episodes

In [ ]:
import importlib, promo_finder as pf; importlib.reload(pf)
pf.MODEL = os.environ['MODEL']; pf.SLEEP = float(os.environ['SLEEP'])
pf.run(PACK, 'outputs')

In [ ]:
# Zip raw outputs for download (your submission's 'raw outputs' - do not edit them)
!zip -qr raw_outputs.zip outputs
from google.colab import files
files.download('raw_outputs.zip')

In [ ]:
import json, collections
rej = json.load(open('outputs/rejected_candidates.json'))
ok = json.load(open('outputs/promos.json'))

# reasons, grouped
def bucket(r):
    r = r["reason"]
    if r.startswith("duration"): return "duration " + ("too short" if float(r.split()[1].rstrip("s")) < 20 else "too long")
    return r.split("(")[0].strip()
print("REJECTION REASONS:", collections.Counter(bucket(r) for r in rej).most_common())

# valid promos per episode (episodes with none)
per = collections.Counter(c["episode"] for c in ok)
print("Episodes with 0 valid:", [e for e in range(1, 51) if e not in per])
print("Segments per valid promo:", collections.Counter(len(c["segments"]) for c in ok))
print("Ends on scene/episode end:", sum(c["ends_at_scene_or_episode_end"] for c in ok), "of", len(ok))

print("\nTOP 10")
for c in ok[:10]:
    print(c["rank"], "Ep", c["episode"], c["duration_s"], "s", [f'{s["start"]}-{s["end"]}' for s in c["segments"]])
    print("   HOOK:", c["hook"][:110]); print("   LAST LINES:", c["last_lines"][:140])

In [ ]:
import os, shutil

# keep run 1 untouched as your "before" baseline
if os.path.exists('outputs') and not os.path.exists('outputs_run1_baseline'):
    os.rename('outputs', 'outputs_run1_baseline')

# new outputs folder that reuses the cached model answers (no re-querying the 50 episodes)
os.makedirs('outputs', exist_ok=True)
if not os.path.exists('outputs/cache'):
    shutil.copytree('outputs_run1_baseline/cache', 'outputs/cache')

print(len(os.listdir('outputs/cache')), "cached episode files copied")

In [ ]:
import json, collections
rej = json.load(open('outputs/rejected_candidates.json'))
ok = json.load(open('outputs/promos.json'))
base = json.load(open('outputs_run1_baseline/promos.json'))

per = collections.Counter(c["episode"] for c in ok)
print("BEFORE:", len(base), "promos | AFTER:", len(ok), "promos (", sum(1 for c in ok if c.get("repaired")), "repaired )")
print("Episodes with 0 valid -> before:", 50 - len({c['episode'] for c in base}), "| after:", [e for e in range(1, 51) if e not in per])
print("Remaining rejections:", collections.Counter(r.get("repair", r["reason"]).split("(")[0][:45] for r in rej).most_common())

print("\nTOP 10 (repaired = R)")
for c in ok[:10]:
    print(c["rank"], "Ep", c["episode"], c["duration_s"], "s", "R" if c.get("repaired") else "")
    print("   HOOK:", c["hook"][:110]); print("   LAST LINES:", c["last_lines"][:150])

print("\nREPAIRED SAMPLE")
for c in [c for c in ok if c.get("repaired")][:5]:
    print("Ep", c["episode"], c["duration_s"], "s |", c["hook_text_first3s"][:120])

In [ ]:
import json
ok = json.load(open('outputs/promos.json'))
print("trimmed:", sum(1 for c in ok if c.get("trimmed")), "| repaired:", sum(1 for c in ok if c.get("repaired")), "| total:", len(ok))